<h1 align="center"><b> Early Detection of Heart Disease using Machine Learning, Deep Learning, NLP & Explainable AI</b></h1>

# The objective of this project is to develop a robust and comprehensive pipeline for the early detection of heart disease using patient clinical data. By leveraging multiple Machine Learning models, Deep Learning techniques, and Natural Language Processing (NLP), the study aims to enhance prediction accuracy and provide meaningful insights. Additionally, model interpretability is ensured through explainable AI techniques, enabling transparency and trust in clinical decision-making. The final system is deployed as an interactive application for real-time prediction and healthcare support.
</p>

# 1. Import  Libraries

In [ ]:
import pandas as pd
import numpy as np
import plotly.express as px
import plotly.graph_objects as go
from plotly.subplots import make_subplots
import plotly.io as pio
pio.renderers.default = 'colab'
from scipy.stats import ttest_ind, chi2_contingency
from sklearn.model_selection import train_test_split, cross_val_score, GridSearchCV
from sklearn.preprocessing import StandardScaler, LabelEncoder
from sklearn.linear_model import LogisticRegression
from sklearn.tree import DecisionTreeClassifier
from sklearn.ensemble import RandomForestClassifier, GradientBoostingClassifier
from sklearn.svm import SVC
from sklearn.neighbors import KNeighborsClassifier
from sklearn.metrics import (accuracy_score, precision_score, recall_score, f1_score,
                             roc_auc_score, roc_curve, confusion_matrix, classification_report)
import xgboost as xgb
import tensorflow as tf
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import Dense, Dropout, BatchNormalization
from tensorflow.keras.optimizers import Adam
from tensorflow.keras.callbacks import EarlyStopping, ReduceLROnPlateau, Callback
from sklearn.feature_extraction.text import TfidfVectorizer
from wordcloud import WordCloud
import matplotlib.pyplot as plt
import shap
import time
import warnings
warnings.filterwarnings('ignore')
import joblib

print(" All libraries imported.")

# 2. Load Dataset

In [ ]:
df = pd.read_csv("/content/NACC_APOE_CVD_filtered (2).csv")

In [ ]:
df.head()

,NACCID,SEX,BIRTHYR,NACCAPOE,DEMENTED,CVHATT,HATTMULT,CVAFIB,CVANGIO,CVBYPASS,...,STROKE,STROKIF,STROKDEC,STKIMAG,CVD,CVDIF,VASC,VASCIF,VASCPS,VASCPSIF
0,NACC000011,2,1944,1.0,0,0.0,NaN,0.0,0.0,0.0,...,0.0,7.0,NaN,NaN,NaN,NaN,0.0,7.0,NaN,NaN
1,NACC000034,2,1935,4.0,0,0.0,8.0,0.0,0.0,0.0,...,NaN,NaN,8.0,8.0,0.0,7.0,NaN,NaN,NaN,NaN
2,NACC000067,1,1952,1.0,0,0.0,NaN,0.0,0.0,0.0,...,0.0,7.0,NaN,NaN,NaN,NaN,0.0,7.0,0.0,7.0
3,NACC000095,1,1926,2.0,1,0.0,NaN,0.0,0.0,0.0,...,0.0,7.0,NaN,NaN,NaN,NaN,0.0,7.0,0.0,7.0
4,NACC000144,1,1930,1.0,0,0.0,NaN,1.0,0.0,0.0,...,0.0,8.0,NaN,NaN,NaN,NaN,8.0,8.0,8.0,8.0


# 3. Data Cleaning & Overview

In [ ]:
df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 40686 entries, 0 to 40685
Data columns (total 43 columns):
 #   Column    Non-Null Count  Dtype  
---  ------    --------------  -----  
 0   NACCID    40686 non-null  object 
 1   SEX       40686 non-null  int64  
 2   BIRTHYR   40686 non-null  int64  
 3   NACCAPOE  40686 non-null  float64
 4   DEMENTED  40686 non-null  int64  
 5   CVHATT    29582 non-null  float64
 6   HATTMULT  7713 non-null   float64
 7   CVAFIB    29536 non-null  float64
 8   CVANGIO   29624 non-null  float64
 9   CVBYPASS  29633 non-null  float64
 10  CVPACDEF  7742 non-null   float64
 11  CVPACE    21901 non-null  float64
 12  CVCHF     29598 non-null  float64
 13  CVANGINA  7738 non-null   float64
 14  CVHVALVE  7738 non-null   float64
 15  CVOTHR    29536 non-null  float64
 16  CVOTHRX   3347 non-null   object 
 17  MYOINF    18764 non-null  float64
 18  CONGHRT   18764 non-null  float64
 19  AFIBRILL  18764 non-null  float64
 20  ANGINA    18764 non-null  fl

In [ ]:
df.shape

(40686, 43)

In [ ]:
df.describe()

,SEX,BIRTHYR,NACCAPOE,DEMENTED,CVHATT,HATTMULT,CVAFIB,CVANGIO,CVBYPASS,CVPACDEF,...,STROKE,STROKIF,STROKDEC,STKIMAG,CVD,CVDIF,VASC,VASCIF,VASCPS,VASCPSIF
count,40686.000000,40686.000000,40686.000000,40686.000000,29582.000000,7713.000000,29536.000000,29624.000000,29633.000000,7742.000000,...,21922.000000,21922.000000,18764.000000,18674.000000,18764.000000,18764.000000,21922.000000,21922.000000,15339.000000,15296.000000
mean,1.565969,1940.892912,1.819471,0.359878,0.110743,7.732141,0.100623,0.118823,0.079101,0.020408,...,0.041602,7.191406,7.852590,7.895041,0.076903,7.128011,2.801387,7.253353,2.945564,7.207178
std,0.495635,12.652103,1.061846,0.479970,0.446419,1.422104,0.365841,0.445972,0.379632,0.158624,...,0.199683,1.032187,1.050306,0.861236,0.266444,1.498858,3.800684,0.903614,3.832178,1.091093
min,1.000000,1896.000000,1.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,...,0.000000,1.000000,0.000000,0.000000,0.000000,1.000000,0.000000,1.000000,0.000000,1.000000
25%,1.000000,1932.000000,1.000000,0.000000,0.000000,8.000000,0.000000,0.000000,0.000000,0.000000,...,0.000000,7.000000,8.000000,8.000000,0.000000,7.000000,0.000000,7.000000,0.000000,7.000000
50%,2.000000,1941.000000,2.000000,0.000000,0.000000,8.000000,0.000000,0.000000,0.000000,0.000000,...,0.000000,7.000000,8.000000,8.000000,0.000000,7.000000,0.000000,7.000000,0.000000,7.000000
75%,2.000000,1949.000000,2.000000,1.000000,0.000000,8.000000,0.000000,0.000000,0.000000,0.000000,...,0.000000,8.000000,8.000000,8.000000,0.000000,8.000000,8.000000,8.000000,8.000000,8.000000
max,2.000000,2003.000000,6.000000,1.000000,2.000000,8.000000,2.000000,2.000000,2.000000,2.000000,...,1.000000,8.000000,8.000000,8.000000,1.000000,8.000000,8.000000,8.000000,8.000000,8.000000


In [ ]:
df.describe().T

,count,mean,std,min,25%,50%,75%,max
SEX,40686.0,1.565969,0.495635,1.0,1.0,2.0,2.0,2.0
BIRTHYR,40686.0,1940.892912,12.652103,1896.0,1932.0,1941.0,1949.0,2003.0
NACCAPOE,40686.0,1.819471,1.061846,1.0,1.0,2.0,2.0,6.0
DEMENTED,40686.0,0.359878,0.479970,0.0,0.0,0.0,1.0,1.0
CVHATT,29582.0,0.110743,0.446419,0.0,0.0,0.0,0.0,2.0
HATTMULT,7713.0,7.732141,1.422104,0.0,8.0,8.0,8.0,8.0
CVAFIB,29536.0,0.100623,0.365841,0.0,0.0,0.0,0.0,2.0
CVANGIO,29624.0,0.118823,0.445972,0.0,0.0,0.0,0.0,2.0
CVBYPASS,29633.0,0.079101,0.379632,0.0,0.0,0.0,0.0,2.0
CVPACDEF,7742.0,0.020408,0.158624,0.0,0.0,0.0,0.0,2.0


In [ ]:
df.isnull().sum()

,0
NACCID,0
SEX,0
BIRTHYR,0
NACCAPOE,0
DEMENTED,0
CVHATT,11104
HATTMULT,32973
CVAFIB,11150
CVANGIO,11062
CVBYPASS,11053


In [ ]:
df.fillna(df.median(numeric_only=True), inplace=True)

In [ ]:
df.isnull().sum()

,0
NACCID,0
SEX,0
BIRTHYR,0
NACCAPOE,0
DEMENTED,0
CVHATT,0
HATTMULT,0
CVAFIB,0
CVANGIO,0
CVBYPASS,0


# 4. Exploratory Data Analysis (EDA)

# 4.1 Target Distribution

In [ ]:
fig = px.pie(df, names='target', title='Heart Disease Distribution (1=Disease, 0=No Disease)',
             color_discrete_sequence=['lightgreen','coral'])
fig.show()

# 4.2 Age Distribution by Target

In [ ]:
fig = px.histogram(df, x='age', color='target', barmode='overlay', opacity=0.6,
                   title='Age Distribution by Heart Disease Status')
fig.show()

# 4.3 Chest Pain Type Analysis

In [ ]:
fig = px.histogram(df, x='cp', color='target', barmode='group',
                   title='Chest Pain Type (cp) by Heart Disease')
fig.show()

# 4.4 Resting Blood Pressure vs Target

In [ ]:
fig = px.box(df, x='target', y='trestbps', color='target',
             title='Resting Blood Pressure (trestbps) by Heart Disease')
fig.show()

# 4.5 Cholesterol Levels by Target

In [ ]:
fig = px.box(df, x='target', y='chol', color='target',
             title='Serum Cholesterol (chol) by Heart Disease')
fig.show()

# 4.6 Maximum Heart Rate Analysis

In [ ]:
fig = px.violin(df, x='target', y='thalach', box=True,
                title='Maximum Heart Rate (thalach) by Heart Disease')
fig.show()

# 4.7 Exercise-Induced Angina

In [ ]:
fig = px.histogram(df, x='exang', color='target', barmode='group',
                   title='Exercise-Induced Angina (exang) by Heart Disease')
fig.show()

# 4.8 ST Depression (Oldpeak) Analysis

In [ ]:
fig = px.box(df, x='target', y='oldpeak', color='target',
             title='ST Depression (oldpeak) by Heart Disease')
fig.show()

# 4.9 Number of Major Vessels

In [ ]:
fig = px.histogram(df, x='ca', color='target', barmode='group',
                   title='Number of Major Vessels (ca)')
fig.show()

# 4.10 Thalassemia Type Analysis

In [ ]:
fig = px.histogram(df, x='thal', color='target', barmode='group',
                   title='Thalassemia Type (thal) Distribution')
fig.show()

# 4.11 Correlation Heatmap

In [ ]:
# Select numerical columns
num_cols = df.select_dtypes(include=np.number).columns
corr = df[num_cols].corr()
fig = px.imshow(corr, text_auto=True, title='Correlation Heatmap',
                color_continuous_scale='RdBu', zmin=-1, zmax=1)
fig.show()

# 4.12 Pairplot of Key Features (Plotly Scatter Matrix)

In [ ]:
key_features = ['age', 'trestbps', 'chol', 'thalach', 'oldpeak', 'target']
fig = px.scatter_matrix(df[key_features], dimensions=key_features[:-1], color='target',
                        title='Pairplot of Key Clinical Features')
fig.update_traces(diagonal_visible=False)
fig.show()

# 5. Statistical Testing

# 5.1 T‑test for Continuous Features

In [ ]:
continuous_features = ['age', 'trestbps', 'chol', 'thalach', 'oldpeak']
ttest_results = []
for col in continuous_features:
    group1 = df[df['target']==1][col]
    group0 = df[df['target']==0][col]
    t_stat, p_val = ttest_ind(group1, group0, equal_var=False)
    ttest_results.append({'Feature': col, 'T-statistic': t_stat, 'P-value': p_val})
ttest_df = pd.DataFrame(ttest_results).sort_values('P-value')
ttest_df['Significant'] = ttest_df['P-value'] < 0.05
print(ttest_df)

# 5.2 Chi‑square Test for Categorical Features

In [ ]:
categorical_features = ['sex', 'cp', 'fbs', 'restecg', 'exang', 'slope', 'ca', 'thal']
chi2_results = []
for col in categorical_features:
    ct = pd.crosstab(df[col], df['target'])
    chi2, p, dof, exp = chi2_contingency(ct)
    chi2_results.append({'Feature': col, 'Chi-square': chi2, 'P-value': p})
chi2_df = pd.DataFrame(chi2_results).sort_values('P-value')
chi2_df['Significant'] = chi2_df['P-value'] < 0.05
print(chi2_df)

# 6. Feature Engineering & Preprocessing

In [ ]:
# Encode categorical variables (if needed)
le = LabelEncoder()
for col in categorical_features:
    if col in df.columns:
        df[col] = le.fit_transform(df[col].astype(str))

In [ ]:
# Define features and target
feature_cols = [col for col in df.columns if col != 'target']
X = df[feature_cols]
y = df['target']

print(f"Features shape: {X.shape}")

In [ ]:
# Train-test split
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42, stratify=y)
print(f"Train size: {X_train.shape[0]}, Test size: {X_test.shape[0]}")

In [ ]:
# Scale features
scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train)
X_test_scaled = scaler.transform(X_test)

# 7. Machine Learning Models

# 7.1 Define All Models

In [ ]:
models = {
    'Logistic Regression': LogisticRegression(max_iter=1000, random_state=42),
    'K-Nearest Neighbors': KNeighborsClassifier(),
    'Decision Tree': DecisionTreeClassifier(random_state=42),
    'Random Forest': RandomForestClassifier(n_estimators=100, random_state=42),
    'SVM': SVC(probability=True, random_state=42),
    'Gradient Boosting': GradientBoostingClassifier(random_state=42),
    'XGBoost': xgb.XGBClassifier(eval_metric='logloss', random_state=42)
}

# 7.2 Train & Evaluate

In [ ]:
results = {}
predictions = {}
probabilities = {}
trained_models = {}

for name, model in models.items():
    model.fit(X_train_scaled, y_train)
    y_pred = model.predict(X_test_scaled)
    y_proba = model.predict_proba(X_test_scaled)[:,1]
    predictions[name] = y_pred
    probabilities[name] = y_proba
    trained_models[name] = model

    acc = accuracy_score(y_test, y_pred)
    prec = precision_score(y_test, y_pred)
    rec = recall_score(y_test, y_pred)
    f1 = f1_score(y_test, y_pred)
    auc = roc_auc_score(y_test, y_proba)

    results[name] = {'Accuracy': acc, 'Precision': prec, 'Recall': rec, 'F1': f1, 'AUC': auc}
    print(f"{name:20} | Acc: {acc:.4f} | AUC: {auc:.4f}")

# 7.3 Model Comparison Bar Chart

In [ ]:
res_df = pd.DataFrame(results).T.reset_index().rename(columns={'index': 'Model'})
fig = px.bar(res_df, x='Model', y=['Accuracy', 'AUC'], barmode='group',
             title='Model Performance Comparison')
fig.show()

# 7.4 Confusion Matrix (Best Model)

In [ ]:
best_model_name = res_df.loc[res_df['AUC'].idxmax(), 'Model']
best_model = trained_models[best_model_name]
y_pred_best = predictions[best_model_name]

cm = confusion_matrix(y_test, y_pred_best)
fig = px.imshow(cm, text_auto=True, title=f'Confusion Matrix – {best_model_name}',
                x=['No Disease', 'Disease'], y=['No Disease', 'Disease'])
fig.show()

# 7.5 ROC Curves for All Models

In [ ]:
fig = go.Figure()
for name, prob in probabilities.items():
    fpr, tpr, _ = roc_curve(y_test, prob)
    auc_val = roc_auc_score(y_test, prob)
    fig.add_trace(go.Scatter(x=fpr, y=tpr, mode='lines', name=f'{name} (AUC={auc_val:.3f})'))
fig.add_trace(go.Scatter(x=[0,1], y=[0,1], mode='lines', name='Random', line=dict(dash='dash')))
fig.update_layout(title='ROC Curves – All Models')
fig.show()

# 7.6 Cross‑Validation

In [ ]:
cv_scores = cross_val_score(best_model, X_train_scaled, y_train, cv=5, scoring='accuracy')
print(f"5-fold CV Accuracy: {cv_scores.mean():.4f} ± {cv_scores.std():.4f}")

# 7.7 Hyperparameter Tuning (GridSearch on Random Forest)

In [ ]:
param_grid = {'n_estimators': [50, 100, 150], 'max_depth': [5, 10, None]}
grid = GridSearchCV(RandomForestClassifier(random_state=42), param_grid, cv=5, scoring='roc_auc')
grid.fit(X_train_scaled, y_train)
print(f"Best parameters: {grid.best_params_}")
print(f"Best CV AUC: {grid.best_score_:.4f}")

# 8. Deep Learning with Epoch Tracking

# 8.1 Custom Epoch Callback

In [ ]:
class EpochLogger(Callback):
    def __init__(self):
        self.epoch_logs = []
        self.start_time = time.time()

    def on_epoch_end(self, epoch, logs=None):
        self.epoch_logs.append({
            'epoch': epoch + 1,
            'loss': logs.get('loss'),
            'accuracy': logs.get('accuracy'),
            'val_loss': logs.get('val_loss'),
            'val_accuracy': logs.get('val_accuracy'),
            'time': time.time() - self.start_time
        })
        if (epoch + 1) % 10 == 0:
            print(f"Epoch {epoch+1}: loss={logs['loss']:.4f}, acc={logs['accuracy']:.4f}, "
                  f"val_loss={logs['val_loss']:.4f}, val_acc={logs['val_accuracy']:.4f}")

epoch_logger = EpochLogger()

# 8.2 Build Neural Network

In [ ]:
nn_model = Sequential([
    Dense(128, activation='relu', input_shape=(X_train_scaled.shape[1],)),
    BatchNormalization(),
    Dropout(0.3),
    Dense(64, activation='relu'),
    BatchNormalization(),
    Dropout(0.3),
    Dense(32, activation='relu'),
    Dense(1, activation='sigmoid')
])

nn_model.compile(optimizer=Adam(learning_rate=0.001),
                 loss='binary_crossentropy',
                 metrics=['accuracy'])

print(nn_model.summary())

# 8.3 Train with Epoch Tracking

In [ ]:
early_stop = EarlyStopping(monitor='val_loss', patience=15, restore_best_weights=True)
lr_reducer = ReduceLROnPlateau(monitor='val_loss', factor=0.5, patience=5)

history = nn_model.fit(X_train_scaled, y_train,
                       epochs=100,
                       batch_size=32,
                       validation_split=0.2,
                       callbacks=[early_stop, lr_reducer, epoch_logger],
                       verbose=0)

print(f"\n Neural Network trained. Completed epochs: {len(history.history['loss'])}")

# 8.4 Plot Training History (Loss & Accuracy)

In [ ]:
fig = make_subplots(rows=1, cols=2, subplot_titles=('Loss', 'Accuracy'))

fig.add_trace(go.Scatter(y=history.history['loss'], name='Train Loss'), row=1, col=1)
fig.add_trace(go.Scatter(y=history.history['val_loss'], name='Validation Loss'), row=1, col=1)
fig.add_trace(go.Scatter(y=history.history['accuracy'], name='Train Accuracy'), row=1, col=2)
fig.add_trace(go.Scatter(y=history.history['val_accuracy'], name='Validation Accuracy'), row=1, col=2)

fig.update_layout(title='Neural Network Training History (Epoch-by-Epoch)')
fig.show()

# 8.5 Evaluate Neural Network

In [ ]:
nn_proba = nn_model.predict(X_test_scaled).flatten()
nn_pred = (nn_proba >= 0.5).astype(int)
nn_acc = accuracy_score(y_test, nn_pred)
nn_auc = roc_auc_score(y_test, nn_proba)

print(f"Neural Network Test Accuracy: {nn_acc:.4f}")
print(f"Neural Network AUC: {nn_auc:.4f}")

# 8.6 Final Epoch Summary

In [ ]:
epoch_logs_df = pd.DataFrame(epoch_logger.epoch_logs)
print("\n Final Epoch Details:")
print(epoch_logs_df.tail(3))
print(f"Total Training Time: {epoch_logs_df['time'].iloc[-1]:.2f} seconds")

# 9. NLP on Feature Names

# 9.1 TF‑IDF Analysis of Feature Names

In [ ]:
feature_text = ' '.join(feature_cols)
vectorizer = TfidfVectorizer(stop_words='english')
tfidf_matrix = vectorizer.fit_transform([feature_text])
tfidf_scores = tfidf_matrix.toarray()[0]
terms = vectorizer.get_feature_names_out()

tfidf_df = pd.DataFrame({'Term': terms, 'Score': tfidf_scores}).sort_values('Score', ascending=False).head(15)
fig = px.bar(tfidf_df, x='Score', y='Term', orientation='h', title='TF‑IDF of Clinical Feature Terms')
fig.show()

# 9.2 Word Cloud of Feature Names

In [ ]:
wordcloud = WordCloud(width=800, height=400, background_color='white').generate(feature_text)
plt.figure(figsize=(10, 5))
plt.imshow(wordcloud, interpolation='bilinear')
plt.title('Feature Names Word Cloud')
plt.axis('off')
plt.show()

# 10. Model Interpretability with SHAP

In [ ]:
# Use Random Forest for SHAP analysis
rf_model = trained_models['Random Forest']
explainer = shap.TreeExplainer(rf_model)
shap_values = explainer.shap_values(X_test_scaled)[1]  # Class 1 (disease)

# Summary plot
shap.summary_plot(shap_values, X_test_scaled, feature_names=feature_cols, show=False)
plt.title('SHAP Summary Plot – Random Forest')
plt.tight_layout()
plt.show()

# Bar plot of mean absolute SHAP values
shap_mean_abs = np.abs(shap_values).mean(axis=0)
shap_df = pd.DataFrame({'Feature': feature_cols, 'SHAP_Importance': shap_mean_abs})
shap_df = shap_df.sort_values('SHAP_Importance', ascending=False).head(10)
fig = px.bar(shap_df, x='SHAP_Importance', y='Feature', orientation='h',
             title='SHAP Feature Importance – Random Forest')
fig.show()

# 11. Save Best Model

In [ ]:
joblib.dump(best_model, 'heart_disease_best_model.pkl')
joblib.dump(scaler, 'heart_scaler.pkl')
joblib.dump(feature_cols, 'heart_features.pkl')
nn_model.save('heart_nn_model.h5')
print(" Models and preprocessing objects saved.")

# 12. Streamlit App Deployment (Interactive Prediction)

In [ ]:
!pip install streamlit pyngrok -q

%%writefile app.py
import streamlit as st
import pandas as pd
import numpy as np
import joblib

# Load model and preprocessors
model = joblib.load('heart_disease_best_model.pkl')
scaler = joblib.load('heart_scaler.pkl')
features = joblib.load('heart_features.pkl')

st.set_page_config(page_title="Heart Disease Predictor", layout="centered")
st.title("🫀 Heart Disease Risk Predictor")
st.markdown("Enter patient health metrics to predict heart disease risk.")

col1, col2 = st.columns(2)
with col1:
    age = st.number_input("Age", 20, 100, 50)
    sex = st.selectbox("Sex", ["Female", "Male"])
    cp = st.selectbox("Chest Pain Type", ["Typical Angina", "Atypical Angina", "Non-anginal Pain", "Asymptomatic"])
    trestbps = st.number_input("Resting Blood Pressure (mm Hg)", 80, 200, 120)
    chol = st.number_input("Cholesterol (mg/dl)", 100, 600, 200)
with col2:
    fbs = st.selectbox("Fasting Blood Sugar >120 mg/dl", ["No", "Yes"])
    restecg = st.selectbox("Resting ECG", ["Normal", "ST-T Abnormality", "Left Ventricular Hypertrophy"])
    thalach = st.number_input("Maximum Heart Rate", 60, 220, 150)
    exang = st.selectbox("Exercise-Induced Angina", ["No", "Yes"])
    oldpeak = st.slider("ST Depression (Oldpeak)", 0.0, 6.0, 1.0)

# Map inputs
sex_val = 1 if sex == "Male" else 0
cp_map = {"Typical Angina": 0, "Atypical Angina": 1, "Non-anginal Pain": 2, "Asymptomatic": 3}
cp_val = cp_map[cp]
fbs_val = 1 if fbs == "Yes" else 0
restecg_map = {"Normal": 0, "ST-T Abnormality": 1, "Left Ventricular Hypertrophy": 2}
restecg_val = restecg_map[restecg]
exang_val = 1 if exang == "Yes" else 0

# Create input vector
input_dict = {
    'age': age, 'sex': sex_val, 'cp': cp_val, 'trestbps': trestbps, 'chol': chol,
    'fbs': fbs_val, 'restecg': restecg_val, 'thalach': thalach, 'exang': exang_val,
    'oldpeak': oldpeak, 'ca': 0, 'thal': 2  # default values
}
input_df = pd.DataFrame([input_dict])[features]
input_scaled = scaler.transform(input_df)
prob = model.predict_proba(input_scaled)[0][1]

st.markdown("---")
if st.button("Predict Heart Disease Risk"):
    st.subheader("Prediction Result")
    if prob >= 0.5:
        st.error(f" High risk of heart disease (Probability: {prob:.2%})")
        st.markdown(" **Recommendation:** Consult a cardiologist for further evaluation.")
    else:
        st.success(f" Low risk of heart disease (Probability: {(1-prob):.2%})")
        st.markdown(" **Recommendation:** Maintain a healthy lifestyle and regular checkups.")

In [ ]:
from pyngrok import ngrok
import subprocess
import time

# Kill any existing Streamlit processes
!pkill streamlit 2>/dev/null
time.sleep(1)

# Run Streamlit in background
process = subprocess.Popen(["streamlit", "run", "app.py", "--server.port", "8501", "--server.headless", "true"])
time.sleep(5)

# Create public URL
public_url = ngrok.connect(8501, "http")
print(f"\n Streamlit app is running!\n Click here to open: {public_url}")
print("Keep this cell running. To stop, interrupt the kernel.")

# Project Completed

# Download the dataset from UCI Heart Disease Repository or use the built‑in version in sklearn.
https://archive.ics.uci.edu/dataset/45/heart+disease